# CETSP results explorer

A single notebook for any results file and any subset of formulations.
Point `SOURCE` at the data, pick formulations with `select(...)`, and every
figure and table below follows.

Nothing is hard-coded to a particular formulation set, instance grid or time
limit: labels are parsed from the `BASE-A-D-cut-S-OC-B` grammar that
`run_experiment.py` builds, the radius regime is recovered from `r_min` and
`r_max`, and the time limit is inferred from the runs that hit it.

The API lives in [`utils/results.py`](utils/results.py) (loading, selection,
tables) and [`utils/plots.py`](utils/plots.py) (figures).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from utils.results import (coverage_report, describe_formulation,
                           formulations_in, load_results, missing_cells,
                           order_formulations, pairwise_table, select,
                           simplify_labels, summary_table, METRICS)
from utils.plots import color_map
from utils.plots import (coverage_heatmap, pairwise_scatter,
                         performance_profile, scaling_plot)

# Uneven coverage between formulations is reported as a warning rather than
# being silently aligned away, so make sure every occurrence is shown.
warnings.simplefilter('always', UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.rcParams['figure.dpi'] = 110

## 1. Load

`load_results` takes one `.csv` or `.xlsx` file. For a workbook it picks the
first sheet that carries the results columns, so pivot-table sheets are
skipped. A file written by an older run carries only nine columns; the rest
are filled with nulls, so both schemas load the same way.

In [ ]:
SOURCE = 'aws-B-S.xlsx'

# Any other results file works the same way, e.g.
#   SOURCE = 'Results/experiment_results.csv'
# Name a sheet with load_results(SOURCE, sheet='...') if the default pick is
# not the one you want.

df = load_results(SOURCE)
print(f'{len(df)} runs from {df.attrs["source"]}')
df.head(3)

## 2. What is in the file

Run this before any comparison: it is the fastest way to see which
formulations are present, how far the instance grid goes, and what is still
missing from a partial run.

In [ ]:
print('Formulations :', len(formulations_in(df)))
for label in formulations_in(df):
    print(f'  {label:<24} {describe_formulation(label)}')

print('\nn            :', sorted(df['n'].unique()))
print('r_mean       :', sorted(df['r_mean'].unique()))
print('sigma        :', sorted(df['sigma'].unique()))
print('Status       :', dict(df['Status'].value_counts()))
print('\nTime limit inferred per size:')
print(df.groupby('n')['TimeLimit'].first())

In [ ]:
report = coverage_report(df)
print(f"instance cells in the grid: {report.attrs['cells']}")
display(report)

gaps = missing_cells(df)
if len(gaps):
    print(f'{len(gaps)} missing runs:')
    display(gaps)

In [ ]:
coverage_heatmap(df, over='n')

## 3. Choosing formulations

`select` filters by what a formulation *is* rather than by its label, so a
selection keeps working when a new modifier is added. Any column works as a
keyword, and scalars and lists are both accepted:

| keyword | meaning |
|---|---|
| `base` | `'ABF'`, `'SBF'`, `'PBF'`, `'BS'` |
| `extended` | `-A`, polyhedral approximation |
| `decomposition` | `-D`, Benders |
| `cut` | `'enum'`, `'dual'`, `'DE'`; `False` for none |
| `strengthen` | `-S`, DFJ strengthening |
| `opt_coeff` | `-OC`, optimised coefficients |
| `symmetry` | `-B`, symmetry breaking |

Every keyword is a filter you opt into, and they combine with AND, so each
one you add narrows the selection. **To keep both values of a modifier, leave
it out** -- omitting `decomposition` keeps the decomposed and the
non-decomposed variants alike. A list says the same thing explicitly, and
works on any keyword: `base=['ABF','SBF']`, `cut=['dual','enum']`,
`n=[20, 25, 30]`. For `cut`, `False` means no cut-type token, and it may
appear inside a list.

A selection that no single predicate describes is the union of two, which
`order_formulations` dedupes and re-sorts.

It returns labels in a stable base-then-modifier order, which is also the
order the colours follow. Pass an explicit list instead whenever that is
easier.

In [ ]:
print('every formulation        :', formulations_in(df))
print()
print('PBF only                 :', select(df, base='PBF'))
print('extended, no decomposition:', select(df, extended=True, decomposition=False))
print('dual cuts                :', select(df, cut='dual'))
print('plain (no modifiers)     :', select(df, extended=False, decomposition=False,
                                           cut=False, opt_coeff=False))

In [ ]:
# Keeping both values of a modifier: omit it, or list both.
print('PBF, -D and not  :', select(df, base='PBF'))
print('same, spelt out  :', select(df, base='PBF', decomposition=[True, False]))
print('PBF, -D only     :', select(df, base='PBF', decomposition=True))
print('dual cuts or none:', select(df, base='PBF', cut=['dual', False]))

# A union, for a set no single predicate describes.
print('ABF -D + PBF non--D:', order_formulations(
    select(df, base='ABF', decomposition=True)
    + select(df, base='PBF', decomposition=False)))

In [ ]:
# PBF decomposed with enumerative cuts is dominated by the dual cut, so it is
# dropped.  `exclude` takes a dict of predicates, or a list of dicts as
# alternatives.  Filtering the frame itself rather than each call keeps it out
# of every figure and table, and keeps the colour ramps as coarse as the kept
# set needs -- a variant nobody plots would otherwise still take a shade.
df = select(df, labels_only=False,
            exclude=dict(base='PBF', extended=False,
                         decomposition=True, cut='enum'))
ALL = formulations_in(df)

# Performance profiles carry exact formulations only.  A monolithic
# approximation (-A without -D) reports a bound that is not valid for the
# CETSP, so putting it on the same axis as an exact method is not a
# like-for-like comparison.
PROFILE = select(df, formulations=ALL,
                 exclude=dict(extended=True, decomposition=False))

print(f'{len(ALL)} formulations, {len(PROFILE)} of them on profiles')
PROFILE

### Labels

`-S`, `-OC` and `-B` are carried by every formulation in these runs, so they
separate nothing and are dropped from the legend. The cut name is dropped per
decomposition, only where it separates nothing: `PBF-D-dual` shortens to
`PBF-D` when no enumerative sibling is in the plot, while `PBF-A-D-dual` keeps
its cut name because `PBF-A-D-enum` is there too. `drop_cut=True`/`False`
overrides that, and `drop=()` keeps every token.

Shortening never merges two formulations into one legend entry: if it would,
the tokens that separate them come back.

In [ ]:
for raw, short in zip(PROFILE, simplify_labels(PROFILE)):
    print(f'{raw:<24} -> {short}')

print()
print('forced off:', simplify_labels(PROFILE, drop_cut=False))
print('no dropping:', simplify_labels(PROFILE, drop=(), drop_cut=False))

# The plots use these automatically; pass labels=[...] for your own.
LABELS = None

### Colours

A formulation keeps its colour wherever it appears. The hue comes from the
base and the shade from the variant's position among *all* that base's
variants in the file, not among the ones in the current plot, so filtering a
figure never repaints the survivors.

Pass `colors=` a `{label: colour}` mapping from `color_map` to pin them
explicitly, or a list to override outright.

In [ ]:
PALETTE = color_map(ALL)
{f: PALETTE[f] for f in PROFILE}

## 4. Performance profiles

The left half counts instances solved to optimality against runtime, the right
half continues with the unsolved ones ordered by final gap, so one step curve
carries a formulation's whole record. Both axes are read off the data.

`title=` sets the heading: on a single panel it titles the axes, and on a
faceted figure it becomes the figure-level title above the per-panel ones.

Coverage that is not uniform raises a warning rather than silently comparing
different instance sets; pass `align=True` to restrict to the cells every
selected formulation ran.

In [ ]:
performance_profile(df, PROFILE, labels=LABELS,
                    title='Instances solved over time and gap',
                    figsize=(7, 5.5))

In [ ]:
# With a 1800 s limit most runs finish in the first few percent of the axis.
# A log time scale usually reads far better; everything else is unchanged.
performance_profile(df, PROFILE, labels=LABELS, time_scale='log',
                    title='Instances solved over time (log) and gap',
                    figsize=(7, 5.5))

### Faceting

`facet_by` takes any column -- `'n'`, `'r_mean'`, `'sigma'` -- or a list of
them for combinations. Panels are laid out automatically, so the
number of values does not matter. `share_scale=True` (the default) keeps one
time limit and gap ceiling across panels so they compare; turn it off to let
each panel use its own range.

In [ ]:
performance_profile(df, PROFILE, labels=LABELS, facet_by='n',
                    time_scale='log', title='By instance size')

In [ ]:
performance_profile(df, PROFILE, labels=LABELS, facet_by='r_mean',
                    time_scale='log', title='By mean radius')

In [ ]:
performance_profile(df, PROFILE, labels=LABELS, facet_by='sigma',
                    time_scale='log', title='By radius variance')

In [ ]:
# Panels can also be named explicitly, with any select() filters behind each.
performance_profile(
    df, PROFILE, labels=LABELS,
    facets={'small instances, tight radii': {'n': [10, 15], 'r_mean': 0.25},
            'large instances, wide radii':  {'n': [25, 30], 'r_mean': 1.0}},
    time_scale='log')

## 5. Summary tables

`metrics` picks the columns from the registry in `utils.results.METRICS`, and
`by` breaks the comparison down by any column. Add an entry to `METRICS` to
make a new column available everywhere.

Note that `Opt` counts runs that proved optimality *of their own model*: for
the approximate formulations (and for B&S) a discretisation gap remains, which
is why `Gap (%)` is reported over all runs and not only the unsolved ones.

In [ ]:
print('available metrics:', list(METRICS))

summary_table(df, ALL, labels=LABELS)

In [ ]:
summary_table(df, ALL, labels=LABELS, by='n',
              metrics=('Runs', 'Opt', 'TimeOpt', 'Gap (%)', 'RootGap (%)',
                       'Nodes'))

In [ ]:
# numeric=True returns raw numbers, for exporting or further arithmetic.
wide = summary_table(df, ALL, by='n', numeric=True,
                     metrics=('Opt',), warn=False)
wide.pivot(index='Formulation', columns='n', values='Opt')

## 6. Head to head

Points are paired on `(n, r_min, r_max, Instance)`, so only cells both
formulations ran are drawn and every point compares the same instance.
`color_by` and `marker_by` accept any column.

In [ ]:
X, Y = PROFILE[0], PROFILE[-1]
pairwise_scatter(df, X, Y, metrics=('Time', 'GapPct'),
                 color_by='r_mean', marker_by='n')

In [ ]:
# The underlying paired frame, for picking out individual instances.
paired = pairwise_table(df, X, Y, metrics=('Time', 'GapPct'))
paired['Time_delta'] = paired['Time_y'] - paired['Time_x']
paired.sort_values('Time_delta').head(10)

## 7. Scaling

Any aggregate of any metric against any column. `agg='solved'` counts
instances solved to optimality.

A line **breaks** wherever a formulation has nothing to aggregate rather than
being carried across the gap, and a warning names the missing points. With
`solved_only=True` that is the common case: a formulation that proved nothing
optimal at some size has no runtime to average there, so its line stops.

That also leaves the surviving means resting on very different numbers of
runs: at n=30 the means below are over 2, 5 and 11 instances. A formulation
can look fastest there simply because it solved only the easy subset. Marker
area tracks the size of the sample behind each point, so a thin mean shrinks
away; the plot is unchanged when every point rests on the same number of
runs. `min_count=` drops the thin ones outright, and `agg='solved'` plots the
counts themselves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
scaling_plot(df, ALL, labels=LABELS, agg='solved', over='n',
             ax=axes[0], title='Instances solved')
scaling_plot(df, ALL, labels=LABELS, metric='GapPct', agg='mean',
             over='n', ax=axes[1], title='Mean gap (%)')
fig.tight_layout()

In [ ]:
scaling_plot(df, ALL, labels=LABELS, metric='Time', agg='mean',
             over='n', solved_only=True, logy=True,
             title='Mean runtime over the instances solved')

In [ ]:
# Same plot, but only where at least 5 instances back the mean.
scaling_plot(df, ALL, labels=LABELS, metric='Time', agg='mean',
             over='n', solved_only=True, logy=True, min_count=5,
             title='Mean runtime, at least 5 instances solved')

## 8. Saving

Every plotting function takes `filename=` and writes a transparent 300 dpi
figure. Tables are ordinary DataFrames, so `.to_csv` and `.to_latex` apply.

In [ ]:
from pathlib import Path

OUT = Path('Results/figures')
OUT.mkdir(parents=True, exist_ok=True)

performance_profile(df, PROFILE, labels=LABELS, facet_by='n',
                    time_scale='log', filename=OUT / 'profile_by_n.png',
                    show=False)
plt.close('all')

table = summary_table(df, ALL, labels=LABELS)
table.to_csv(OUT / 'summary.csv', index=False)
print(table.to_latex(index=False))